In [1]:
"""
Project: ANAC Aerodrome Database Restructuring (Post-2024 Regulatory Shift)

Objective:
In early 2024, ANAC regulatory changes significantly altered the demand for traditional 
aeronautical services. This pipeline was developed to restructure the sales funnel 
for the Basic Protection Zone Plan (PBZP) by automating lead prospecting and 
qualification through government data mining.

Data Management:
- Historical Data: The 'data/raw/' directory contains the 2024 records used during 
  the original project (Public Aerodromes, Private Aerodromes, and Helipads).
- Updated Sources: For real-time updates, official ANAC Open Data portals are used:
  * Public: https://www.gov.br/anac/pt-br/acesso-a-informacao/dados-abertos/areas-de-atuacao/aerodromos/aerodromos-publicos
  * Private: https://www.gov.br/anac/pt-br/acesso-a-informacao/dados-abertos/areas-de-atuacao/aerodromos/aerodromos-privados

Methodology:
1. Ingestion: Loading standardized CSV records (UTF-8 encoded).
2. Validation: Web Scraping the DECEA AGA Portal to check current documentation status.
3. Classification: Categorizing leads by commercial viability (Status 0: No active plan; 
   Status 2: Inconsistencies).
4. Export: Generation of 'leads_pbzp_final.csv' for CRM/Notion integration.

Author: André Gambogi (AEROJR. Solutions)
"""

import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
import os

# Environment initialization
print("=" * 80)
print("MARKET INTELLIGENCE PIPELINE: INITIALIZED")
print("Data Source: ANAC/DECEA Open Data (2024 Historical Base)")
print("=" * 80)

MARKET INTELLIGENCE PIPELINE: INITIALIZED
Data Source: ANAC/DECEA Open Data (2024 Historical Base)


In [2]:
# ==============================================================================
# DATA CONSOLIDATION & PRE-PROCESSING
# ==============================================================================
# This section loads the standardized UTF-8 databases and consolidates them 
# into a single master DataFrame. All records are kept, as the search engine 
# can query by Name, ICAO (OACI), or CIAD.

# Defining directory paths
RAW_DIR = "data/raw/"
PROCESSED_DIR = "data/processed/"

if not os.path.exists(PROCESSED_DIR):
    os.makedirs(PROCESSED_DIR)

# File names as standardized in the data/raw folder
files = {
    "Private Aerodrome": "aerodromos_privados.csv",
    "Public Aerodrome": "aerodromos_publicos.csv",
    "Helipad": "helipontos.csv"
}

dfs = []

for category, filename in files.items():
    file_path = os.path.join(RAW_DIR, filename)
    try:
        temp_df = pd.read_csv(file_path, sep = ";")
        temp_df['Category'] = category
        dfs.append(temp_df)
        print(f"LOADED: {filename} ({len(temp_df)} records)")
    except FileNotFoundError:
        print(f"WARNING: {filename} not found. Skipping...")

# Concatenating all databases
df_master = pd.concat(dfs, ignore_index=True)

# Initializing tracking columns
# Status_PBZP: Categorized result from the AGA Portal
# Card_Created: Flag to manage lead processing and prevent duplicates
df_master['Status_PBZP'] = "Pending"
df_master['Card_Created'] = False

print("-" * 80)
print(f"CONSOLIDATION COMPLETE: {len(df_master)} total leads ready for validation.")
print("-" * 80)

LOADED: aerodromos_privados.csv (3429 records)
LOADED: aerodromos_publicos.csv (497 records)
LOADED: helipontos.csv (1459 records)
--------------------------------------------------------------------------------
CONSOLIDATION COMPLETE: 5385 total leads ready for validation.
--------------------------------------------------------------------------------


In [12]:
# ================================================================================
# BLOCK 3: LEAD VALIDATION ENGINE (DECEA API INTEGRATION)
# This block defines the core automation logic to query the DECEA SysAGA API.
# It validates if an aerodrome has a registered Protection Zone Plan (PBZP).
# ================================================================================


def validate_pbzp_status(icao_code, ciad_code=None, name=None):
    """
    Queries the DECEA (SysAGA) API to check for PBZP plans.
    Executes a cascading search strategy: ICAO -> CIAD -> Name.
    
    Args:
        icao_code (str): The ICAO designator (e.g., 'SNRF').
        ciad_code (str): The CIAD registration number.
        name (str): The aerodrome denomination name.
        
    Returns:
        int: Status Code (0: No plan found, 1: Validated, 2: Ambiguous/Multiple, -1: Error)
    """
    
    # New API Endpoint for the SysAGA Portal (Updated 2024+)
    url = "https://sysaga.decea.mil.br/api/portal/planos?page=1"

    # Modern headers to properly authenticate the JSON POST request and bypass blocks
    headers = {
        "Content-Type": "application/json",
        "Accept": "application/json, text/plain, */*",
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
    }

    # Define strict priority sequence for search keys
    potential_keys = [icao_code, ciad_code, name]
    
    # Filter out null/empty values to only try valid keys
    valid_keys = [str(k).strip() for k in potential_keys if pd.notna(k) and str(k).strip() != ""]

    if not valid_keys:
        return 0

    # Cascading Search Logic
    for search_key in valid_keys:
        payload = {
            "aerodromo": search_key,
            "estado": "",
            "municipio": [],
            "quantidade": 5,
            "tipo": ""
        }

        try:
            # Executing the POST request with the updated JSON payload
            response = requests.post(url, json=payload, headers=headers, timeout=15)
            response.raise_for_status()
            
            # Process the new JSON response structure
            data = response.json()
            results = data.get("downloads", {}).get("data", [])

            # Business Logic for Qualification:
            if len(results) > 1:
                return 2  # Ambiguous data (Multiple matches found)
            elif len(results) == 1:
                return 1  # Lead is validated (Plan exists)
            
            # Se len(results) == 0, o código não retorna nada e o loop 'for' 
            # avança automaticamente para testar a próxima chave (CIAD, depois Nome).

        except requests.exceptions.RequestException as e:
            print(f"CONNECTION ERROR: Failed to validate '{search_key}'. Reason: {e}")
            return -1

    # If the loop finishes trying ICAO, CIAD, and Name, and ALL of them returned 0 results:
    return 0


print("-" * 80)
print("VALIDATION ENGINE: Logic compiled and ready for batch processing.")
print("-" * 80)

--------------------------------------------------------------------------------
VALIDATION ENGINE: Logic compiled and ready for batch processing.
--------------------------------------------------------------------------------


In [13]:
# ================================================================================
# BLOCK 4: BATCH PROCESSING & INCREMENTAL CHECKPOINTING
# This block iterates through the consolidated dataframe to perform validation.
# It includes a checkpoint mechanism to ensure data persistence during long 
# execution cycles or potential network interruptions.
# ================================================================================


# Filter only for leads that haven't been processed yet ('Pending' status)
# This allows the pipeline to be idempotent and resume from where it left off
pending_leads = df_master[df_master['Status_PBZP'] == "Pending"]
total_pending = len(pending_leads)

print(f"INITIATING BATCH PROCESSING: {total_pending} records to validate.")

# Configuration for incremental saving
SAVE_INTERVAL = 20  # Save every 20 records to balance I/O operations and safety
processed_counter = 0

# Mapping results to a structured business classification
# This simplifies future filtering in the Sales Funnel/CRM
status_mapping = {
    0: "Qualified: No PBZP Found",
    1: "Unqualified: PBZP Already Exists",
    2: "Review Required: Ambiguous Results",
    -1: "System Error: Connection Failed"
}

for index, row in pending_leads.iterrows():
    # Primary identifiers extracted from ANAC/DECEA database structure
    icao = row.get('Designador ICAO')
    ciad = row.get('Código CIAD')
    name = row.get('Nome')

    # Execute the scraping/validation logic defined in Block 3
    result_code = validate_pbzp_status(icao_code=icao, ciad_code=ciad, name=name)
    
    # Determine the string status based on the mapping
    status_text = status_mapping.get(result_code, "Unknown")
    
    # Update the master dataframe with the classified status
    df_master.at[index, 'Status_PBZP'] = status_text
    
    processed_counter += 1

    # Real-time console tracking
    search_key = icao if pd.notna(icao) else name
    print(f"[{processed_counter}/{total_pending}] Validating: {search_key} -> Status: {status_text}")

    # Incremental Checkpoint Logic
    if processed_counter % SAVE_INTERVAL == 0:
        checkpoint_path =  PROCESSED_DIR + "LEADS_PBZP_CHECKPOINT.csv"
        # Using utf-8-sig to ensure Excel compatibility for the end-user
        df_master.to_csv(checkpoint_path, index=False, sep=";", encoding="utf-8-sig")
        print(f"--- CHECKPOINT REACHED: {processed_counter} leads processed and saved. ---")

    # Strategic delay (Politeness Policy)
    # Prevents IP flagging and ensures stability of the government portal connection
    time.sleep(0.8)

# Final export of the fully processed dataset
OUTPUT_FILE = PROCESSED_DIR + "LEADS_PBZP_QUALIFIED.csv"
df_master.to_csv(OUTPUT_FILE, index=False, sep=";", encoding="utf-8-sig")

print("-" * 80)
print(f"PROCESSING COMPLETE: {processed_counter} records analyzed.")
print(f"Final output generated: {OUTPUT_FILE}")
print("-" * 80)

INITIATING BATCH PROCESSING: 5318 records to validate.
[1/5318] Validating: Fazenda Marrecão -> Status: Unqualified: PBZP Already Exists
[2/5318] Validating: Fazenda Perobal -> Status: Unqualified: PBZP Already Exists
[3/5318] Validating: Fazenda Quatrirmãs -> Status: Unqualified: PBZP Already Exists
[4/5318] Validating: Fazenda Nossa Senhora das Graças -> Status: Review Required: Ambiguous Results
[5/5318] Validating: Fazenda Três Minas -> Status: Unqualified: PBZP Already Exists
[6/5318] Validating: Estância Hércules -> Status: Unqualified: PBZP Already Exists
[7/5318] Validating: Fazenda Tangará -> Status: Review Required: Ambiguous Results
[8/5318] Validating: Aeroclube São José do Rio Pardo -> Status: Unqualified: PBZP Already Exists
[9/5318] Validating: FAZENDA BANDEIRANTES -> Status: Review Required: Ambiguous Results
[10/5318] Validating: Caracaranã -> Status: Qualified: No PBZP Found


KeyboardInterrupt: 

In [11]:
# ================================================================================
# BLOCK 5: DATA ANALYSIS, INSIGHTS & FINAL EXPORT
# This final block generates a summary report of the prospecting results and 
# performs the final cleanup of the dataset for commercial delivery.
# ================================================================================

# 1. Statistical Summary of Prospecting Efforts
print("=" * 80)
print("PROSPECTING SUMMARY REPORT")
print("=" * 80)

# Calculate distribution of Lead Status
status_distribution = df_master['Status_PBZP'].value_counts()
total_leads = len(df_master)

for status, count in status_distribution.items():
    percentage = (count / total_leads) * 100
    print(f"{status:35} | Count: {count:4} | ({percentage:.2f}%)")

# 2. Lead Qualification KPI
# Focused on "Qualified: No PBZP Found" - the primary target for sales
qualified_leads_count = status_distribution.get("Qualified: No PBZP Found", 0)
conversion_potential = (qualified_leads_count / total_leads) * 100

print("-" * 80)
print(f"MARKET POTENTIAL: {qualified_leads_count} aerodromes require new PBZP compliance.")
print(f"CONVERSION RATE POTENTIAL: {conversion_potential:.2f}% of the database is actionable.")
print("-" * 80)

# 3. Final Data Cleanup
# Removing technical flags that are not needed for the final business report
cols_to_remove = ['Card_Created'] # Example of internal control column
final_delivery_df = df_master.drop(columns=[col for col in cols_to_remove if col in df_master.columns])

# 4. Final Export - High Compatibility Format
# Using the 'Qualified' suffix to distinguish from raw data
final_filename = PROCESSED_DIR + "FINAL_QUALIFIED_PBZP_MARKET_LEADS.csv"
final_delivery_df.to_csv(final_filename, index=False, sep=";", encoding="utf-8-sig")

print(f"SUCCESS: Final dataset exported to {final_filename}")
print("PIPELINE EXECUTION COMPLETED.")
print("=" * 80)

PROSPECTING SUMMARY REPORT
Pending                             | Count: 5318 | (98.76%)
Unqualified: PBZP Already Exists    | Count:   38 | (0.71%)
Review Required: Ambiguous Results  | Count:   20 | (0.37%)
Qualified: No PBZP Found            | Count:    6 | (0.11%)
System Error: Connection Failed     | Count:    3 | (0.06%)
--------------------------------------------------------------------------------
MARKET POTENTIAL: 6 aerodromes require new PBZP compliance.
CONVERSION RATE POTENTIAL: 0.11% of the database is actionable.
--------------------------------------------------------------------------------
SUCCESS: Final dataset exported to data/processed/FINAL_QUALIFIED_PBZP_MARKET_LEADS.csv
PIPELINE EXECUTION COMPLETED.
